**Building the Infectious Disease Adjuvant Knowledge Base (KB)** directly from your merged dataset.
**STEP 1 — Build the Adjuvant KB**
### What this code does:

* Loads your finalized infectious dataset
* Groups rows by **`c_adjuvant_vo_id`** (canonical adjuvant concept)
* Extracts:

  * preferred names
  * synonyms (merged from multiple columns)
  * PMIDs
  * vaccine names
  * pathogens
  * descriptions
* Produces a **KB DataFrame**: one row per canonical adjuvant

In [1]:
import pandas as pd
import re
from collections import defaultdict
from utils import extract_pmids_from_row

from utils import fix_mojibake


# -------------------------
# 1. Load infectious dataset
# -------------------------
df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", encoding="utf-8-sig")


# Build allowed synonym set from vaxjo (run once)
vaxjo = pd.read_csv("Dataset/VIOLIN_12-10-2025/raw/t_vaxjo_202512101311_clean.csv", encoding="utf-8-sig")
allowed = set(vaxjo["c_vaxjo_name"].dropna().str.strip())
allowed |= set(
    vaxjo["c_alternative_names"].dropna()
         .str.split(";")
         .explode()
         .str.strip()
         .dropna()
)

# -------------------------
# 2. Helper function: clean text
# -------------------------
def clean_text(x):
    if pd.isna(x): 
        return None
    x = fix_mojibake(str(x)).strip()
    return x if x != "" else None

# -------------------------
# Prepare PMID regex + text columns
# -------------------------
'''PMID_RE = re.compile(r"\[PubMed:\s*(\d{5,10})\s*\]", flags=re.I)
TEXT_COLS = df.select_dtypes(include=["object", "string"]).columns.tolist()

def extract_pmids_from_text(row):
    pmids = set()
    for col in TEXT_COLS:
        val = row[col]
        if isinstance(val, str):
            found = PMID_RE.findall(val)
            if found:
                pmids.update(found)
    return pmids'''

TEXT_COLS = df.select_dtypes(include=["object", "string"]).columns.tolist()

def extract_pmids_from_text(row):
    return set(extract_pmids_from_row(row, TEXT_COLS))


# -------------------------
# 3. Initialize KB structure
# -------------------------
kb_records = []

grouped = df.groupby("c_adjuvant_vo_id")

for vo_id, g in grouped:
    
    # --- Preferred name ---
    preferred_name = (
        g['c_vaxjo_name'].dropna().unique().tolist() +
        g['c_adjuvant_label'].dropna().unique().tolist()
    )
    preferred_name = preferred_name[0] if preferred_name else None

    # --- Synonyms ---
    synonyms = set()
    for col in ['c_vaxjo_name', 'c_adjuvant_label']:#, 'c_adjuvant']:
        vals = g[col].dropna().unique().tolist()
        synonyms.update([clean_text(v) for v in vals if clean_text(v)])

    # Alternative names
    if "c_alternative_names" in g.columns:
        for alt in g["c_alternative_names"].dropna():
            parts = [clean_text(x) for x in str(alt).split(";")]
            synonyms.update([p for p in parts if p])

    # Keep only official VO names/alt names
    #synonyms = [s for s in synonyms if s in allowed]

    # Keep only official VO names/alt names, plus c_adjuvant_label
    allowed_labels = set(g["c_adjuvant_label"].dropna().str.strip())

    synonyms = [s for s in synonyms if (s in allowed or s in allowed_labels)]


    synonyms = sorted(list(synonyms))


    # --- PMIDs: extract from ALL text columns ---
    all_pmids = set()
    for idx, row in g.iterrows():
        all_pmids.update(extract_pmids_from_text(row))

    # --- Vaccine, pathogen, disease context ---
    vaccines = sorted(g["c_vaccine_name"].dropna().unique().tolist())
    pathogens = sorted(g["c_pathogen_name"].dropna().unique().tolist())
    diseases = sorted(g["c_disease_name"].dropna().unique().tolist())

    # --- Descriptions ---
    descriptions = (
        g["c_adjuvant_description"].dropna().unique().tolist() +
        g["c_description"].dropna().unique().tolist()
    )
    descriptions = list({clean_text(d) for d in descriptions if clean_text(d)})

    # --- Add KB record ---
    kb_records.append({
        "adjuvant_vo_id": vo_id,
        "preferred_name": preferred_name,
        "synonyms": synonyms,
        "num_synonyms": len(synonyms),
        "pmids": sorted(list(all_pmids)),
        "num_pmids": len(all_pmids),
        "vaccine_contexts": vaccines,
        "num_vaccines": len(vaccines),
        "pathogen_contexts": pathogens,
        "disease_contexts": diseases,
        "description_list": descriptions
    })


# -------------------------
# 4. Create KB DataFrame
# -------------------------
kb_df = pd.DataFrame(kb_records)
kb_df = kb_df.sort_values("preferred_name")

# -------------------------
# 5. Save to disk
# -------------------------
kb_df.to_json("Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.json", 
              orient="records", indent=2, force_ascii=False)

kb_df.to_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.csv", index=False, encoding="utf-8-sig")

kb_df.head()


<>:38: SyntaxWarning: invalid escape sequence '\['
<>:38: SyntaxWarning: invalid escape sequence '\['
C:\Users\hasin.rehana\AppData\Local\Temp\ipykernel_87636\1421717561.py:38: SyntaxWarning: invalid escape sequence '\['
  '''PMID_RE = re.compile(r"\[PubMed:\s*(\d{5,10})\s*\]", flags=re.I)


,adjuvant_vo_id,preferred_name,synonyms,num_synonyms,pmids,num_pmids,vaccine_contexts,num_vaccines,pathogen_contexts,disease_contexts,description_list
96,VO_0005271,2B182C,[2B182C],1,"[10196265, 10201942, 10403592, 10573147, 10775...",72,[Seasonal Influenza Vaccine -- Influenza A str...,1,[Influenza virus],[Influenza (flu)],[]
101,VO_0005303,2F52,[2F52],1,"[10196265, 10201942, 10403592, 10573147, 10775...",72,[Human Influenza Vaccine],1,[Influenza virus],[Influenza (flu)],[]
76,VO_0001330,AF03,[AF03],1,"[10196265, 10201942, 10403592, 10573147, 10775...",71,[Inactivated H1N1(2009) Vaccine with AF03 adju...,1,[Influenza virus],[Influenza (flu)],[A proprietary oil-in-water adjuvant of Sanofi...
33,VO_0001264,AS-2 vaccine adjuvant,"[AS-2 vaccine adjuvant, AS02, SBAS2, SmithKlin...",3,"[10076914, 10085007, 10225931, 10456931, 10506...",173,"[FMP1/AS02A, M. tuberculosis Mtb72F Protein Su...",4,"[Mycobacterium tuberculosis, Plasmodium spp.]","[Malaria, Tuberculosis]",[The Plasmodium falciparum vaccine candidate F...
70,VO_0001320,AS03,[AS03],1,"[10196265, 10201942, 10403592, 10573147, 10775...",72,"[AREPANRIX H1N1, Pandemrix]",2,[Influenza virus],[Influenza (flu)],[AS03 adjuvant composed of squalene (10.69 mil...


In [2]:
import pandas as pd
from collections import Counter
import ast

print("=== DATASET-LEVEL STATS ===")
print(f"Total rows (infectious triples): {len(df)}")
print(f"Unique vaccines: {df['c_vaccine_name'].nunique()}")
print(f"Unique pathogens: {df['c_pathogen_name'].nunique()}")
print(f"Unique disease names: {df['c_disease_name'].nunique()}")
print(f"Unique adjuvant VO IDs: {df['c_adjuvant_vo_id'].nunique()}")

print("\n=== ADJUVANT KB STATS ===")
print(f"Total unique adjuvant concepts: {kb_df['adjuvant_vo_id'].nunique()}")
print(f"Average synonyms per adjuvant: {kb_df['num_synonyms'].mean():.2f}")
print(f"Max synonyms for any adjuvant: {kb_df['num_synonyms'].max()}")
print(f"Min synonyms for any adjuvant: {kb_df['num_synonyms'].min()}")

print("\n=== SYNONYM COVERAGE DISTRIBUTION ===")
print(kb_df['num_synonyms'].describe())

print("\n=== VACCINE CONTEXT STATS ===")
print(f"Average vaccines per adjuvant: {kb_df['num_vaccines'].mean():.2f}")
print(f"Max vaccines for any adjuvant: {kb_df['num_vaccines'].max()}")

print("\n=== PATHOGEN CONTEXT STATS ===")
print(f"Unique pathogen contexts across KB: {kb_df['pathogen_contexts'].explode().nunique()}")
print(f"Average pathogens per adjuvant: {kb_df['pathogen_contexts'].apply(len).mean():.2f}")

print("\n=== DISEASE CONTEXT STATS ===")
print(f"Unique disease contexts across KB: {kb_df['disease_contexts'].explode().nunique()}")
print(f"Average diseases per adjuvant: {kb_df['disease_contexts'].apply(len).mean():.2f}")

print("\n=== TEXT COVERAGE STATS ===")
text_cols = [
    'c_full_text_pathogen',
    'c_full_text',
    'c_adjuvant_description',
    'c_description_vaccine',
    'c_description'
]

for col in text_cols:
    print(f"{col}: {df[col].notna().sum()} non-null ({df[col].notna().mean()*100:.1f}%)")

print("\n=== PMID STATS ===")
pmid_cols = [c for c in df.columns if "pmid" in c.lower()]
total_pmids = set()
for col in pmid_cols:
    total_pmids.update(df[col].dropna().astype(str).tolist())

print(f"Total unique PMIDs across all columns: {len(total_pmids)}")

print("\n=== TOP 20 MOST FREQUENT ADJUVANTS IN INFECTIOUS DATASET ===")
print(df['c_vaxjo_name'].value_counts().head(20))

print("\n=== TOP 20 MOST FREQUENT PATHOGENS ===")
print(df['c_pathogen_name'].value_counts().head(20))

print("\n=== TOP 20 MOST FREQUENT DISEASE NAMES ===")
print(df['c_disease_name'].value_counts().head(20))


=== DATASET-LEVEL STATS ===
Total rows (infectious triples): 546
Unique vaccines: 377
Unique pathogens: 82
Unique disease names: 78
Unique adjuvant VO IDs: 103

=== ADJUVANT KB STATS ===
Total unique adjuvant concepts: 103
Average synonyms per adjuvant: 1.78
Max synonyms for any adjuvant: 7
Min synonyms for any adjuvant: 0

=== SYNONYM COVERAGE DISTRIBUTION ===
count    103.000000
mean       1.776699
std        1.236155
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        7.000000
Name: num_synonyms, dtype: float64

=== VACCINE CONTEXT STATS ===
Average vaccines per adjuvant: 4.25
Max vaccines for any adjuvant: 50

=== PATHOGEN CONTEXT STATS ===
Unique pathogen contexts across KB: 82
Average pathogens per adjuvant: 2.71

=== DISEASE CONTEXT STATS ===
Unique disease contexts across KB: 78
Average diseases per adjuvant: 2.61

=== TEXT COVERAGE STATS ===
c_full_text_pathogen: 545 non-null (99.8%)
c_full_text: 493 non-null (90.3%)
c_adjuvant_descriptio

In [3]:
df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", encoding="utf-8-sig")
print(df[df['c_organism_type'] == 'other'][['c_pathogen_name','c_disease_name']].drop_duplicates())

Empty DataFrame
Columns: [c_pathogen_name, c_disease_name]
Index: []


In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", encoding="utf-8-sig")

# Normalize organism type
df["c_organism_type_clean"] = (
    df["c_organism_type"].fillna("")
    .str.strip()
    .str.lower()
)

infectious_types = ["g- bacterium", "g+ bacterium", "virus", "parasite", "fungus"]

# List top 20 adjuvants
top20 = (
    df["c_vaxjo_name"]
    .value_counts()
    .head(20)
    .index
    .tolist()
)

print("\n=== TOP 20 ADJUVANTS VALIDATION ===\n")

problem_rows = []  # will collect any suspicious items

for adj in top20:
    subset = df[df["c_vaxjo_name"] == adj]

    pathogens = sorted(subset["c_pathogen_name"].dropna().unique().tolist())
    diseases = sorted(subset["c_disease_name"].dropna().unique().tolist())
    org_types = sorted(subset["c_organism_type_clean"].dropna().unique().tolist())

    print(f"\n--- {adj} ---")
    print("Pathogens:", pathogens)
    print("Diseases:", diseases)
    print("Organism types:", org_types)

    # Identify any non-infectious organism types
    bad = [t for t in org_types if t not in infectious_types]
    if bad:
        print("WARNING: Non-infectious organism types found:", bad)
        problem_rows.append((adj, bad))

# Summary
print("\n\n=== SUMMARY ===")
if len(problem_rows) == 0:
    print("All top 20 adjuvants are associated ONLY with infectious pathogens.")
else:
    print("Check required for the following:")
    for p in problem_rows:
        print(p)



=== TOP 20 ADJUVANTS VALIDATION ===


--- aluminum hydroxide vaccine adjuvant ---
Pathogens: ['Bordetella pertussis', 'Clostridium botulinum', 'Clostridium tetani', 'Corynebacterium diphtheriae', 'Escherichia coli', 'Haemophilus influenzae', 'Helicobacter pylori', 'Hepatitis A virus', 'Hepatitis B virus', 'Human Respiratory Syncytial Virus', 'Japanese encephalitis virus', 'Mumps virus', 'Mycobacterium tuberculosis', 'Mycoplasma gallisepticum', 'Neisseria meningitidis', 'Plasmodium spp.', 'Poliovirus', 'Pseudomonas aeruginosa', 'Salmonella spp.', 'Streptococcus pneumoniae', 'Tick-borne Encephalitis Virus (TBEV)']
Diseases: ['Botulism', 'Chronic respiratory disease', 'Diphtheria', 'Hemorrhagic colitis', 'Hepatitis A', 'Hepatitis B', 'Japanese encephalitis', 'Malaria', 'Meningitis', 'Mumps', 'Pneumonia', 'Polio', 'Pseudomonas aeruginosa infection', 'Respiratory tract disease', 'Salmonellosis', 'Tetanus', 'Tick-borne Encephalitis', 'Tuberculosis', 'Ulcers', 'Whooping Cough']
Organism type

In [5]:
[col for col in df.columns if "pmid" in col.lower()]


[]

In [6]:
import pandas as pd
import re

# Load cleaned dataset with PMIDs
df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", encoding="utf-8-sig")

# Regex for extracting PMIDs
PMID_RE = re.compile(r"\[PubMed:\s*(\d{5,10})\s*\]", flags=re.I)

# Select only text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

stats = []

for col in text_cols:
    col_values = df[col].astype(str)

    # Extract all PMIDs in the column
    all_pmids = col_values.apply(lambda x: PMID_RE.findall(x))

    # Flatten list of lists → single list
    pmid_list = [pmid for sublist in all_pmids for pmid in sublist]

    if len(pmid_list) == 0:
        continue  # skip empty columns

    stats.append({
        "column": col,
        "total_pmids": len(pmid_list),
        "unique_pmids": len(set(pmid_list))
    })

# Convert to DataFrame and sort
pmid_stats_df = pd.DataFrame(stats)
pmid_stats_df = pmid_stats_df.sort_values("total_pmids", ascending=False)

print("\n📊 Column-level PMID stats:\n")
print(pmid_stats_df.to_string(index=False))



📊 Column-level PMID stats:

                column  total_pmids  unique_pmids
  c_full_text_pathogen        71023          1211
           c_full_text         2122           290
        c_pathogenesis          707            49
        c_introduction          659            50
         c_description          495            45
 c_protective_immunity          481            52
            c_function          411            45
          c_host_range          408            40
              c_safety          276            25
         c_preparation          252            21
              c_dosage          248            28
            c_adjuvant          214           136
             c_antigen          193           124
          c_components          160            22
 c_preparation_vaccine          137            87
c_adjuvant_description          134            87
             c_storage          113             1
           c_structure          101             1
 c_description_vaccin

In [7]:
import pandas as pd
import re

# Load KB and dataset
kb_df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.csv")
df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", encoding="utf-8-sig")

# Regex for PMID extraction
PMID_RE = re.compile(r"\[PubMed:\s*(\d{5,10})\s*\]", flags=re.I)

text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

column_stats = []

for col in text_cols:
    col_values = df[col].astype(str)

    # Extract PMIDs from this column
    all_pmids = col_values.apply(lambda x: PMID_RE.findall(x))

    # Flatten into list
    pmid_list = [pmid for sublist in all_pmids for pmid in sublist]

    if len(pmid_list) == 0:
        continue

    column_stats.append({
        "column": col,
        "total_pmids": len(pmid_list),
        "unique_pmids": len(set(pmid_list)),
        "coverage_in_kb": len(set(pmid_list) & set().union(*kb_df["pmids"].apply(eval)))
    })

col_stats_df = pd.DataFrame(column_stats)
col_stats_df = col_stats_df.sort_values("total_pmids", ascending=False)

print("\n📊 KB-Based Column-level PMID Stats:\n")
print(col_stats_df.to_string(index=False))



📊 KB-Based Column-level PMID Stats:

                column  total_pmids  unique_pmids  coverage_in_kb
  c_full_text_pathogen        71023          1211            1211
           c_full_text         2122           290             290
        c_pathogenesis          707            49              49
        c_introduction          659            50              50
         c_description          495            45              45
 c_protective_immunity          481            52              52
            c_function          411            45              45
          c_host_range          408            40              40
              c_safety          276            25              25
         c_preparation          252            21              21
              c_dosage          248            28              28
            c_adjuvant          214           136             136
             c_antigen          193           124             124
          c_components          160   